In [1]:
import duckdb

conn = duckdb.connect("../tmp/polymarket.db")
conn.execute("PRAGMA threads=8")
conn.execute("PRAGMA enable_object_cache=true")

market_path = "../Data/data/polymarket/markets/markets_*.parquet"
trade_path = "../Data/data/polymarket/trades/trades_*.parquet"
block_path = "../Data/data/polymarket/blocks/blocks_*.parquet"

In [2]:
# drop existing table if it exists
conn.execute("DROP TABLE IF EXISTS trader_counts")

conn.execute(f"""
    CREATE TABLE trader_counts AS
    SELECT
        trader,
        COUNT(*) AS trade_count
    FROM (
        SELECT maker AS trader FROM '{trade_path}'
        UNION ALL
        SELECT taker AS trader FROM '{trade_path}'
    ) t
    WHERE trader IS NOT NULL
    GROUP BY trader
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [8]:
conn.execute(f"""SELECT COUNT(*) FROM trader_counts WHERE trade_count >= 100""").fetchone()

(479487,)

In [9]:
N = 1000 # keep ~1/N of traders
MIN_TRADES = 100

conn.execute("DROP TABLE IF EXISTS traders_sampled")
conn.execute("DROP TABLE IF EXISTS trades_sampled")

# sample 1 / N traders with at least MIN_TRADES trades
conn.execute(f"""
    CREATE TABLE traders_sampled AS
    SELECT trader
    FROM trader_counts
    WHERE trade_count >= {MIN_TRADES}
    AND ABS(HASH(trader)) % {N} = 0
""")

# select trades where either maker or taker is in the sampled_traders
conn.execute(f"""
    CREATE TABLE trades_sampled AS
    SELECT *
    FROM '{trade_path}'
    WHERE maker IN (SELECT trader FROM traders_sampled)
    OR taker IN (SELECT trader FROM traders_sampled)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [10]:
# save trades_sampled
conn.execute(f"COPY trades_sampled TO '../samples/trades_sampled.csv' (HEADER, DELIMITER ',')")
conn.execute(f"COPY trades_sampled TO '../samples/trades_sampled.parquet' (FORMAT 'parquet')")

# save traders_sampled
conn.execute(f"COPY traders_sampled TO '../samples/traders_sampled.csv' (HEADER, DELIMITER ',')")
conn.execute(f"COPY traders_sampled TO '../samples/traders_sampled.parquet' (FORMAT 'parquet')")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [11]:
conn.execute("SELECT COUNT(*) FROM trades_sampled").fetchone()

(305708,)

In [12]:
threshold = 0.5

market_query = f"""
CREATE TABLE markets
AS SELECT
    question,
    clob_token_ids ->> '$[0]' AS yes_token,
    clob_token_ids ->> '$[1]' AS no_token,
    CAST(outcome_prices ->> '$[0]' AS FLOAT) AS yes_price,
    CAST(outcome_prices ->> '$[1]' AS FLOAT) AS no_price
FROM '{market_path}';
CREATE INDEX idx_yes_token ON markets(yes_token);
CREATE INDEX idx_no_token ON markets(no_token);
"""

tokens_query = f"""
CREATE TABLE tokens
AS
SELECT
    yes_token AS token,
    CASE
        WHEN yes_price > {(100 - threshold) / 100.0} AND no_price < {threshold / 100.0} THEN TRUE
        WHEN yes_price < {threshold / 100.0} AND no_price > {(100 - threshold) / 100.0} THEN FALSE
        ELSE NULL
    END AS implied_outcome,
    CASE
        WHEN (yes_price > {(100 - threshold) / 100.0} AND no_price < {threshold / 100.0})
          OR (yes_price < {threshold / 100.0} AND no_price > {(100 - threshold) / 100.0}) THEN TRUE
        ELSE FALSE
    END AS resolved,
    1 = 1 as token_type
FROM markets

UNION ALL

SELECT
    no_token AS token,
    CASE
        WHEN yes_price > {(100 - threshold) / 100.0} AND no_price < {threshold / 100.0} THEN FALSE
        WHEN yes_price < {threshold / 100.0} AND no_price > {(100 - threshold) / 100.0} THEN TRUE
        ELSE NULL
    END AS implied_outcome,
    CASE
        WHEN (yes_price > {(100 - threshold) / 100.0} AND no_price < {threshold / 100.0})
          OR (yes_price < {threshold / 100.0} AND no_price > {(100 - threshold) / 100.0}) THEN TRUE
        ELSE FALSE
    END AS resolved,
    1 = 0 as token_type
FROM markets;
CREATE INDEX idx_token ON tokens(token);
"""

conn.execute("DROP TABLE IF EXISTS markets")
conn.execute(market_query)
conn.execute("DROP TABLE IF EXISTS tokens")
conn.execute(tokens_query)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))